# Week 58, Day 2: Growing, pruning and the instability of trees

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w058/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(582)

def cells(n, g, flip=0.1):
    y = g.integers(0, 3, n)
    width = np.array([0.25, 0.45, 0.65])[y] + 0.07 * g.normal(size=n)
    rate = np.array([25, 10, 4])[y] * np.exp(0.35 * g.normal(size=n))
    burst = np.array([0.05, 0.2, 0.4])[y] + 0.1 * g.normal(size=n)
    noisy = np.where(g.random(n) < flip, g.integers(0, 3, n), y)
    return np.c_[width, rate, burst, g.normal(size=n), g.random(n)], noisy

def gini_counts(c):
    n = c.sum(-1, keepdims=True); return 1 - np.sum((c / np.maximum(n, 1)) ** 2, axis=-1)

def best_split(X, y, K, feats=None, min_leaf=1):
    best = (None, None, np.inf); n = len(y)
    for j in (range(X.shape[1]) if feats is None else feats):
        o = np.argsort(X[:, j]); xs, ys = X[o, j], y[o]
        left = np.cumsum(np.eye(K)[ys], axis=0)[:-1]; right = left[-1] + np.eye(K)[ys[-1]] - left
        nl = np.arange(1, n); nr = n - nl
        imp = (nl * gini_counts(left) + nr * gini_counts(right)) / n
        ok_ = (xs[1:] > xs[:-1]) & (nl >= min_leaf) & (nr >= min_leaf)
        imp = np.where(ok_, imp, np.inf); i = int(np.argmin(imp))
        if imp[i] < best[2]: best = (j, (xs[i] + xs[i + 1]) / 2, imp[i])
    return best

def grow(X, y, K, depth=99, min_leaf=1, feats_fn=None):
    counts = np.bincount(y, minlength=K)
    if depth == 0 or len(y) < 2 * min_leaf or counts.max() == len(y): return ("leaf", counts)
    j, t, imp = best_split(X, y, K, None if feats_fn is None else feats_fn(), min_leaf)
    if j is None or imp >= gini_counts(counts): return ("leaf", counts)
    m = X[:, j] <= t
    return ("node", j, t, grow(X[m], y[m], K, depth - 1, min_leaf, feats_fn), grow(X[~m], y[~m], K, depth - 1, min_leaf, feats_fn), counts)

def predict_proba(tree, X, K):
    if tree[0] == "leaf": return np.tile(tree[1] / tree[1].sum(), (len(X), 1))
    _, j, t, L, R, _c = tree; m = X[:, j] <= t; out = np.zeros((len(X), K))
    out[m], out[~m] = predict_proba(L, X[m], K), predict_proba(R, X[~m], K); return out

def n_leaves(tree):
    return 1 if tree[0] == "leaf" else n_leaves(tree[3]) + n_leaves(tree[4])

X, y = cells(300, r); X_te, y_te = cells(3000, r)
acc = lambda tree, X_, y_: np.mean(predict_proba(tree, X_, 3).argmax(1) == y_)
print("ready: 300 training neurons, 3,000 test neurons, 10% of labels flipped")

### Exercise: Error against depth

Grow trees of depth 1, 2, 3, 4, 6, 8 and unlimited (99), and record training accuracy, test accuracy and the number of leaves.

In [ ]:
depths = [1, 2, 3, 4, 6, 8, 99]
tr_acc, te_acc, leaves = [], [], []
for d in depths:
    tree = grow(X, y, 3, depth=d)
    tr_acc.append(...); te_acc.append(...); leaves.append(n_leaves(tree))
    print(f"depth {d:2d}: {leaves[-1]:3d} leaves, training accuracy {tr_acc[-1]:.3f}, test accuracy {te_acc[-1]:.3f}")

In [ ]:
import numpy as np
def test_train():
    "training accuracy rises with depth and is near 1 for the unlimited tree"
    assert tr_acc[-1] > 0.97 and tr_acc[-1] >= tr_acc[0]
def test_test():
    "the unlimited tree is worse on test data than the best depth"
    assert te_acc[-1] < max(te_acc) - 0.03

test_train()
test_test()
print("All checks passed")

### Exercise: Depth and leaf size by five-fold cross-validation

Compute five-fold cross-validated accuracy for each combination of depth and minimum leaf size. Refit the best combination on all training data and report its test accuracy, and the best test accuracy on the grid for comparison.

In [ ]:
cv_depths, min_leaves = [2, 3, 4, 6, 99], [1, 5, 15, 30]
folds = np.array_split(r.permutation(300), 5)
cv = np.zeros((len(cv_depths), len(min_leaves)))
for i, d in enumerate(cv_depths):
    for j, ml in enumerate(min_leaves):
        s = 0
        for te in folds:
            tr = np.setdiff1d(np.arange(300), te)
            s += ...
        cv[i, j] = s / 300
i, j = np.unravel_index(np.argmax(cv), cv.shape)
te_chosen = acc(grow(X, y, 3, cv_depths[i], min_leaves[j]), X_te, y_te)
te_best = max(acc(grow(X, y, 3, d, ml), X_te, y_te) for d in cv_depths for ml in min_leaves)
print(np.round(cv, 3))
print(f"chosen depth {cv_depths[i]}, minimum leaf {min_leaves[j]}: test accuracy {te_chosen:.3f} (best on grid {te_best:.3f})")

In [ ]:
import numpy as np
def test_grid():
    "cv has one entry per depth and minimum leaf size"
    assert cv.shape == (len(cv_depths), len(min_leaves))
def test_choice():
    "the chosen tree's test accuracy is within 0.03 of the best on the grid"
    assert te_chosen >= te_best - 0.03

test_grid()
test_choice()
print("All checks passed")

### Exercise: Reduced-error pruning

Complete prune(tree, Xv, yv): prune the children first, then replace the node by a leaf (with the node's training counts) if that leaf classifies the validation examples reaching this node at least as well as the subtree.

In [ ]:
perm = r.permutation(300); tr_i, va_i = perm[:200], perm[200:]
full = grow(X[tr_i], y[tr_i], 3)

def prune(tree, Xv, yv):
    if tree[0] == "leaf" or len(yv) == 0:
        return tree if tree[0] == "leaf" else ("leaf", tree[5])
    _, j, t, L, R, counts = tree
    m = Xv[:, j] <= t
    L, R = prune(L, Xv[m], yv[m]), prune(R, Xv[~m], yv[~m])
    node = ("node", j, t, L, R, counts)
    as_leaf = ...
    as_node = np.sum(predict_proba(node, Xv, 3).argmax(1) == yv)
    return ("leaf", counts) if as_leaf >= as_node else node

pruned = prune(full, X[va_i], y[va_i])
print(f"full tree: {n_leaves(full)} leaves, test accuracy {acc(full, X_te, y_te):.3f}")
print(f"pruned:    {n_leaves(pruned)} leaves, test accuracy {acc(pruned, X_te, y_te):.3f}")

In [ ]:
import numpy as np
def test_smaller():
    "pruning removes most of the leaves of the full tree"
    assert n_leaves(pruned) < 0.5 * n_leaves(full)
def test_better():
    "the pruned tree is more accurate on test data than the full tree"
    assert acc(pruned, X_te, y_te) > acc(full, X_te, y_te)

test_smaller()
test_better()
print("All checks passed")

### Exercise: Twenty trees from one dataset

Grow 20 trees on bootstrap resamples, at depth 2 and at unlimited depth. Record each deep tree's root feature and threshold, and compute the average fraction of test neurons on which pairs of trees disagree, for shallow and for deep trees.

In [ ]:
root_feats, root_thr, preds_shallow, preds_deep = [], [], [], []
for b in range(20):
    idx = r.integers(0, 300, 300)
    deep = grow(X[idx], y[idx], 3)
    shallow = grow(X[idx], y[idx], 3, depth=2)
    root_feats.append(deep[1]); root_thr.append(deep[2])
    preds_deep.append(predict_proba(deep, X_te, 3).argmax(1)); preds_shallow.append(predict_proba(shallow, X_te, 3).argmax(1))

def mean_disagreement(P):
    P = np.array(P)
    return ...

disagree_shallow, disagree_deep = mean_disagreement(preds_shallow), mean_disagreement(preds_deep)
print("root features of the 20 deep trees:", root_feats)
w_thr = [t for f_, t in zip(root_feats, root_thr) if f_ == 0]
print(f"original tree root: feature {grow(X, y, 3)[1]}; spike-width thresholds at the root: {min(w_thr):.3f} to {max(w_thr):.3f} ms")
print(f"pairwise disagreement on test neurons: depth 2 {disagree_shallow:.3f}, unlimited {disagree_deep:.3f}")

In [ ]:
import numpy as np
def test_roots():
    "root_feats holds the root feature of each of the 20 bootstrap trees"
    assert len(root_feats) == 20
def test_disagree():
    "deep bootstrap trees disagree with each other on more test neurons than shallow ones do"
    assert disagree_deep > disagree_shallow

test_roots()
test_disagree()
print("All checks passed")